# SDCP-v2 on Wikipedia KB (Fair Comparison)

**Goal**: Run SDCP-v2 using the same Wikipedia KB as Li et al. (articles_l3.pkl, 999 articles).

- chunk_size=64, overlap=8 (matching Li et al. default)
- Datasets: TruthfulQA (615Q) + MMLU (1596Q)
- This allows **direct fair comparison** with Li et al. baselines

**If kernel is alive from SDCP_v2_AlwaysUncertain.ipynb, skip Cell 0.**

In [1]:
# ── Cell 0: Setup (skip if model already loaded in memory) ────────────────────
import os, sys, gc, time, json
import numpy as np, pandas as pd
import torch, faiss, psutil
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer as rs_module
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
from tqdm import tqdm

BASE_DIR     = '/home/user/RAG_Best_Practices/RAG_best_practices-main'
MODELS_DIR   = '/home/user/RAG_Best_Practices/models'
DATASETS_DIR = '/home/user/RAG_Best_Practices/datasets'
OUTPUT_DIR   = '/home/user/RAG_Best_Practices/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_TRUTHFULQA     = 615
MMLU_PER_SUBJECT = 28
RANDOM_SEED      = 42
CHOICE_LABELS    = ['A', 'B', 'C', 'D']
INST_S = '[INST]'
INST_E = '[/INST]'
SYS    = 'You are a truthful expert question-answering bot and should correctly and concisely answer the following question'

def show_mem():
    ram  = psutil.virtual_memory()
    vram = torch.cuda.memory_allocated()/1024**3 if torch.cuda.is_available() else 0
    print(f'  RAM {ram.used/1024**3:.1f}/{ram.total/1024**3:.1f}GB | VRAM {vram:.1f}GB')

gc.collect(); torch.cuda.empty_cache(); show_mem()

MODEL_PATH = f'{MODELS_DIR}/mistral-7b'
EMBED_PATH = f'{MODELS_DIR}/minilm'

print('Loading Mistral-7B (4bit)...')
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
llm = AutoModelForCausalLM.from_pretrained(MODEL_PATH, quantization_config=bnb, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
show_mem()

print('Loading MiniLM...')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
embed_model = SentenceTransformer(EMBED_PATH).to(DEVICE)
show_mem()
print('Models ready!')

  RAM 11.5/47.0GB | VRAM 0.0GB
Loading Mistral-7B (4bit)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

  RAM 12.2/47.0GB | VRAM 4.2GB
Loading MiniLM...
  RAM 12.2/47.0GB | VRAM 4.3GB
Models ready!


In [2]:
# ── Cell 1: Load datasets (same splits as SDCP-v2) ────────────────────────────
# TruthfulQA
print('Loading TruthfulQA...')
tqa_raw = load_from_disk(f'{DATASETS_DIR}/truthfulqa').to_pandas()
tqa_all = tqa_raw[['question','best_answer','correct_answers','incorrect_answers']].copy()
tqa_all['correct_answers']   = tqa_all['correct_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['incorrect_answers'] = tqa_all['incorrect_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['best_answer']       = tqa_all['best_answer'].apply(lambda x: [x] if x else [])
tqa_all = tqa_all[(tqa_all['correct_answers'].apply(len) > 1) &
                   (tqa_all['incorrect_answers'].apply(len) > 1)].reset_index(drop=True)
tqa_test_idx = tqa_all.sample(n=N_TRUTHFULQA, random_state=RANDOM_SEED).index
tqa = tqa_all.loc[tqa_test_idx].reset_index(drop=True)
del tqa_raw; gc.collect()
print(f'  TruthfulQA test: {len(tqa)}Q')

# MMLU
print('Loading MMLU...')
mmlu_raw = load_from_disk(f'{DATASETS_DIR}/mmlu').to_pandas()

def mmlu_to_unified(row):
    choices   = list(row['choices'])
    ans_idx   = int(row['answer'])
    correct   = choices[ans_idx]
    incorrect = [choices[i] for i in range(len(choices)) if i != ans_idx]
    formatted_q = (row['question'] + '\n' +
                   '\n'.join(f'{CHOICE_LABELS[i]}) {choices[i]}' for i in range(len(choices))))
    return pd.Series({'question': formatted_q, 'question_plain': row['question'],
                      'subject': row['subject'], 'best_answer': [correct],
                      'correct_answers': [correct], 'incorrect_answers': incorrect,
                      'answer_idx': ans_idx, 'choices': choices})

mmlu_test_parts = []
for subject, group in mmlu_raw.groupby('subject'):
    group = group.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n_test = min(MMLU_PER_SUBJECT, len(group))
    mmlu_test_parts.append(group.iloc[:n_test])
mmlu = pd.concat(mmlu_test_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
del mmlu_raw; gc.collect()
print(f'  MMLU test: {len(mmlu)}Q ({mmlu["subject"].nunique()} subjects)')

print('Datasets ready!')

Loading TruthfulQA...
  TruthfulQA test: 615Q
Loading MMLU...
  MMLU test: 1596Q (57 subjects)
Datasets ready!


In [3]:
# ── Cell 2: Build Wikipedia KB index (same chunking as Li et al.) ─────────────
# Parameters matching Li et al. base_config
CHUNK_SIZE = 64   # tokens
OVERLAP    = 8    # tokens

print('Loading Wikipedia KB (articles_l3.pkl)...')
wiki_kb = pd.read_pickle(f'{BASE_DIR}/resources/articles_l3.pkl')
print(f'  {len(wiki_kb)} articles loaded')
print(f'  Columns: {list(wiki_kb.columns)}')

# Chunk articles using same tokenizer as Li et al.
print('Chunking articles (chunk_size=64, overlap=8)...')
chunks = []
step = max(1, CHUNK_SIZE - OVERLAP)
for doc_id, row in wiki_kb.iterrows():
    text  = str(row['text_en'])
    title = str(row['title_en'])
    tokens = tokenizer.encode(text)
    for start in range(0, len(tokens), step):
        end = min(start + CHUNK_SIZE, len(tokens))
        chunk_text = tokenizer.decode(tokens[start:end], skip_special_tokens=True)
        chunks.append({'text': chunk_text, 'title': title, 'org_doc_id': doc_id})
        if end == len(tokens):
            break

wiki_chunks = pd.DataFrame(chunks)
print(f'  {len(wiki_chunks)} chunks from {len(wiki_kb)} articles')

# Build FAISS index
print('Building FAISS index...')
chunk_texts = wiki_chunks['text'].tolist()
chunk_embs  = embed_model.encode(chunk_texts, show_progress_bar=True, batch_size=256)
chunk_embs  = np.array(chunk_embs, dtype=np.float32)
faiss.normalize_L2(chunk_embs)
wiki_index = faiss.IndexFlatIP(chunk_embs.shape[1])
wiki_index.add(chunk_embs)
print(f'  FAISS index: {wiki_index.ntotal} vectors')
show_mem()

Loading Wikipedia KB (articles_l3.pkl)...
  999 articles loaded
  Columns: ['title_en', 'text_en', 'title_de', 'text_de', 'title_fr', 'text_fr', 'description_en', 'aliases_en', 'description_de', 'aliases_de', 'description_fr', 'aliases_fr']
Chunking articles (chunk_size=64, overlap=8)...
  213211 chunks from 999 articles
Building FAISS index...


Batches:   0%|          | 0/833 [00:00<?, ?it/s]

  FAISS index: 213211 vectors
  RAM 15.1/47.0GB | VRAM 4.3GB


In [4]:
# ── Cell 3: SDCP-v2 core functions ───────────────────────────────────────────
SDCP_PARAMS = {
    'alpha': 0.45, 'beta': 0.35, 'gamma': 0.20,
    'top_k_retrieve': 15, 'top_k_context': 4,
    'max_gen_tokens': 25, 'max_pos_tokens': 20, 'max_neg_tokens': 20,
}

def generate(prompts, max_new_tokens=25, num_beams=2):
    enc = tokenizer(prompts, return_tensors='pt', padding=True,
                    truncation=True, max_length=2048).to(DEVICE)
    input_len = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = llm.generate(input_ids=enc['input_ids'],
                           attention_mask=enc['attention_mask'],
                           max_new_tokens=max_new_tokens,
                           num_beams=num_beams,
                           pad_token_id=tokenizer.eos_token_id)
    return [tokenizer.decode(r[input_len:], skip_special_tokens=True).strip()
            or 'I have no comment' for r in out]

def get_token_probs(prompt, max_new_tokens=20):
    enc = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                           return_dict_in_generate=True, output_scores=True,
                           pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out.sequences[0][enc['input_ids'].shape[1]:],
                             skip_special_tokens=True).strip()
    cert = 0.0
    if out.scores:
        probs = torch.softmax(out.scores[0][0], dim=-1)
        top2  = torch.topk(probs, 2).values
        cert  = (top2[0] - top2[1]).item()
    return text, cert

def clean_response(resp):
    for stop in ['\nQuestion:','\nQ:','\n---','\nIncorrect','\nCorrect',
                 '\nVERIFIED','\nExample','\n\n','\nMy initial','\nContext:',
                 '\nFor the question', '\nCommon']:
        if stop in resp: resp = resp[:resp.index(stop)]
    return resp.strip().strip('"').strip("'") or 'I have no comment'

def retrieve_wiki(query, p_pos, p_neg, k_retrieve=15, k_context=4,
                  alpha=0.45, beta=0.35, gamma=0.20):
    """Retrieve from Wikipedia KB using SDCP contrastive scoring."""
    q_emb   = np.array(embed_model.encode([query],  show_progress_bar=False), dtype=np.float32)
    pos_emb = np.array(embed_model.encode([p_pos],  show_progress_bar=False), dtype=np.float32)
    neg_emb = np.array(embed_model.encode([p_neg],  show_progress_bar=False), dtype=np.float32) if p_neg else None
    faiss.normalize_L2(q_emb)

    _, idxs = wiki_index.search(q_emb, k_retrieve)
    retrieved = wiki_chunks.iloc[idxs[0]].reset_index(drop=True)

    sentences  = retrieved['text'].tolist()
    s_embs     = embed_model.encode(sentences, show_progress_bar=False)
    q_sims     = cosine_similarity(s_embs, q_emb).flatten()
    p_sims     = cosine_similarity(s_embs, pos_emb).flatten()
    n_sims     = (cosine_similarity(s_embs, neg_emb).flatten()
                  if neg_emb is not None else np.zeros(len(sentences)))
    scores     = alpha * q_sims + beta * p_sims - gamma * n_sims
    top_idx    = np.argsort(-scores)[:k_context]
    context    = ' '.join([sentences[i] for i in top_idx])
    return context

def compute_metrics(generated, references):
    scorer = rs_module.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    r1s, ecss = [], []
    for gen, refs in zip(generated, references):
        best_r1 = 0
        for ref in refs:
            if not ref: continue
            s = scorer.score(ref, gen)
            best_r1 = max(best_r1, s['rouge1'].fmeasure)
        r1s.append(best_r1 * 100)
        try:
            embs = embed_model.encode([refs[0], gen])
            ecss.append(float(cosine_similarity([embs[0]], [embs[1]])[0][0]) * 100)
        except:
            ecss.append(0.0)
    return np.mean(r1s), np.mean(ecss)

def compute_accuracy(generated, dataset):
    correct = 0
    for gen, (_, row) in zip(generated, dataset.iterrows()):
        correct_text = row['best_answer'][0] if isinstance(row['best_answer'], list) else row['best_answer']
        ans_idx = int(row.get('answer_idx', -1))
        if ans_idx >= 0:
            label = CHOICE_LABELS[ans_idx]
            if label in gen[:5] or correct_text.lower() in gen.lower(): correct += 1
        else:
            if correct_text.lower() in gen.lower(): correct += 1
    return correct / len(generated) * 100

print('Core functions ready.')

Core functions ready.


In [5]:
# ── Cell 4: Run SDCP-v2 with Wikipedia KB on TruthfulQA ──────────────────────
print('=== SDCP-v2 | Wikipedia KB | TruthfulQA | test=615Q ===')
print(f'    Using {wiki_index.ntotal} Wikipedia chunks as KB')

t0 = time.time()
generated_tqa, references_tqa, prior_log_tqa = [], [], []

for idx, row in tqdm(tqa.iterrows(), total=len(tqa), desc='SDCP-v2-Wiki/TQA'):
    query       = row['question']
    best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

    # Step 1: self-distill priors
    pos_prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer concisely:{INST_E}'
    p_pos, cert = get_token_probs(pos_prompt, max_new_tokens=20)
    p_pos = clean_response(p_pos)

    neg_prompt = (f'{INST_S}What is a common misconception or false belief '
                  f'that people hold about this topic?\n'
                  f'Question: {query}\nCommon wrong belief (very short):{INST_E}')
    p_neg = clean_response(generate([neg_prompt], max_new_tokens=20)[0])

    # Step 2: contrastive retrieval from Wikipedia KB
    context = retrieve_wiki(query, p_pos, p_neg)

    # Step 3: generate with full uncertain-path prompt
    prompt = (f'{INST_S}{SYS}\nRetrieved context: {context}\n'
              f'My initial thought: {p_pos}\n'
              f'Common mistake to avoid: {p_neg}\n'
              f'Question: {query}\nVerified answer:{INST_E}')
    final = clean_response(generate([prompt], max_new_tokens=25)[0])

    generated_tqa.append(final)
    references_tqa.append(best_answer)
    prior_log_tqa.append({'p_pos': p_pos, 'p_neg': p_neg, 'cert': cert})

    if idx % 30 == 0: gc.collect(); torch.cuda.empty_cache()

r1_tqa, ecs_tqa = compute_metrics(generated_tqa, references_tqa)
elapsed = (time.time() - t0) / 60
print(f'\nTruthfulQA Results:')
print(f'  R1={r1_tqa:.2f}  ECS={ecs_tqa:.2f}')
print(f'Elapsed: {elapsed:.1f} min')

=== SDCP-v2 | Wikipedia KB | TruthfulQA | test=615Q ===
    Using 213211 Wikipedia chunks as KB


SDCP-v2-Wiki/TQA: 100%|██████████| 615/615 [48:14<00:00,  4.71s/it]



TruthfulQA Results:
  R1=32.46  ECS=64.44
Elapsed: 48.3 min


In [6]:
# ── Cell 5: Run SDCP-v2 with Wikipedia KB on MMLU ────────────────────────────
print('=== SDCP-v2 | Wikipedia KB | MMLU | test=1596Q ===')

t0 = time.time()
generated_mmlu, references_mmlu, prior_log_mmlu = [], [], []

for idx, row in tqdm(mmlu.iterrows(), total=len(mmlu), desc='SDCP-v2-Wiki/MMLU'):
    query       = row['question']
    best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

    # Step 1: self-distill priors
    pos_prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer concisely:{INST_E}'
    p_pos, cert = get_token_probs(pos_prompt, max_new_tokens=20)
    p_pos = clean_response(p_pos)

    q_plain = query.split('\n')[0]
    neg_prompt = (f'{INST_S}What is a plausible but INCORRECT answer that students '
                  f'commonly give for this type of question?\n'
                  f'Question: {q_plain}\nCommon wrong answer (very short):{INST_E}')
    p_neg = clean_response(generate([neg_prompt], max_new_tokens=20)[0])

    # Step 2: contrastive retrieval from Wikipedia KB
    context = retrieve_wiki(query, p_pos, p_neg)

    # Step 3: generate with full uncertain-path prompt
    prompt = (f'{INST_S}{SYS}\nRetrieved context: {context}\n'
              f'My initial thought: {p_pos}\n'
              f'Common mistake to avoid: {p_neg}\n'
              f'Question: {query}\nVerified answer:{INST_E}')
    final = clean_response(generate([prompt], max_new_tokens=25)[0])

    generated_mmlu.append(final)
    references_mmlu.append(best_answer)
    prior_log_mmlu.append({'p_pos': p_pos, 'p_neg': p_neg, 'cert': cert})

    if idx % 30 == 0: gc.collect(); torch.cuda.empty_cache()

r1_mmlu, ecs_mmlu = compute_metrics(generated_mmlu, references_mmlu)
acc_mmlu = compute_accuracy(generated_mmlu, mmlu)
elapsed = (time.time() - t0) / 60
print(f'\nMMLU Results:')
print(f'  R1={r1_mmlu:.2f}  ECS={ecs_mmlu:.2f}  Acc={acc_mmlu:.1f}%')
print(f'Elapsed: {elapsed:.1f} min')

=== SDCP-v2 | Wikipedia KB | MMLU | test=1596Q ===


SDCP-v2-Wiki/MMLU: 100%|██████████| 1596/1596 [2:07:50<00:00,  4.81s/it] 



MMLU Results:
  R1=38.40  ECS=53.57  Acc=53.3%
Elapsed: 128.0 min


In [7]:
# ── Cell 6: Summary and comparison with Li et al. ────────────────────────────
print('=' * 70)
print('COMPARISON: SDCP-v2 (Wikipedia KB) vs. Li et al. best variants')
print('=' * 70)
print(f'\n{"Method":<30} {"TQA R1":>8} {"MMLU R1":>8} {"MMLU Acc":>10}')
print('-' * 60)

# Li et al. results (from Table 2 of their paper)
li_results = [
    ('Li et al. Base',       26.81, 10.42, 'N/A'),
    ('Li et al. ICL1Doc+',   30.62, 25.87, 'N/A'),
    ('Li et al. ICL2Doc+',   30.24, 26.01, 'N/A'),
    ('Li et al. 80Doc80S',   28.85, 10.69, 'N/A'),
]
for name, tqa, mmlu_r1, acc in li_results:
    print(f'{name:<30} {tqa:>8.2f} {mmlu_r1:>8.2f} {str(acc):>10}')

print('-' * 60)
# SDCP-v2 with Wikipedia KB (same KB as Li et al.)
print(f'{"SDCP-v2 (Wikipedia KB)":<30} {r1_tqa:>8.2f} {r1_mmlu:>8.2f} {acc_mmlu:>9.1f}%')

# SDCP-v2 with same-dataset KB (our previous result)
print(f'{"SDCP-v2 (dataset KB)":<30} {35.59:>8.2f} {35.44:>8.2f} {"45.0%":>10}')
print('=' * 70)

# Save results
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
summary = {
    'SDCP_v2_WikiKB_TQA':  {'R1': round(r1_tqa, 3),  'ECS': round(ecs_tqa, 3)},
    'SDCP_v2_WikiKB_MMLU': {'R1': round(r1_mmlu, 3), 'ECS': round(ecs_mmlu, 3), 'Accuracy': round(acc_mmlu, 3)},
}
out_path = f'{OUTPUT_DIR}/sdcp_v2_wiki_kb_{ts}.json'
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2)

# Save CSVs
pd.DataFrame({'generated': generated_tqa,
              'reference': [r[0] for r in references_tqa],
              'p_pos': [l['p_pos'] for l in prior_log_tqa],
              'p_neg': [l['p_neg'] for l in prior_log_tqa]}
            ).to_csv(f'{OUTPUT_DIR}/sdcp_v2_wiki_tqa_{ts}.csv', index=False)

pd.DataFrame({'generated': generated_mmlu,
              'reference': [r[0] for r in references_mmlu],
              'p_pos': [l['p_pos'] for l in prior_log_mmlu],
              'p_neg': [l['p_neg'] for l in prior_log_mmlu]}
            ).to_csv(f'{OUTPUT_DIR}/sdcp_v2_wiki_mmlu_{ts}.csv', index=False)

print(f'\nSaved to: {out_path}')

COMPARISON: SDCP-v2 (Wikipedia KB) vs. Li et al. best variants

Method                           TQA R1  MMLU R1   MMLU Acc
------------------------------------------------------------
Li et al. Base                    26.81    10.42        N/A
Li et al. ICL1Doc+                30.62    25.87        N/A
Li et al. ICL2Doc+                30.24    26.01        N/A
Li et al. 80Doc80S                28.85    10.69        N/A
------------------------------------------------------------
SDCP-v2 (Wikipedia KB)            32.46    38.40      53.3%
SDCP-v2 (dataset KB)              35.59    35.44      45.0%

Saved to: /home/user/RAG_Best_Practices/outputs/sdcp_v2_wiki_kb_20260506_175450.json
